In [ ]:
%pip install -q pandas scikit-learn
print('Dependências instaladas no Kernel do Jupyter!')

# Tarefa 2: Baseline e Rotulagem
Este notebook executa o cálculo do baseline por fluxo (apenas Período A), calcula as métricas relativas (Período B) e aplica a árvore de decisão baseada em regras do RFC (OK, RISCO, FALHA).

In [1]:
import pandas as pd
import numpy as np

# 1. Leitura do bruto (NÃO DEVE MODIFICAR ESTE ARQUIVO)
df_raw = pd.read_csv('../data/interim/tabela_bruta_completa.csv')
print(f"Linhas totais: {len(df_raw)}")

# Ordenar por fluxo e timestamp
df_raw.sort_values(by=['fluxo_id', 'timestamp'], inplace=True)
display(df_raw.head())

Linhas totais: 347515


,timestamp,measurement_id,probe_id,dst_addr,fluxo_id,avg,min,max,sent,rcvd,perda_pct,jitter_ms,timeout_atual,periodo
143163,1.788680e+09,9333345.0,6251.0,193.0.0.164,6251|193.0.0.164,28.080205,27.994538,28.164329,3.0,3.0,0.0,0.084906,0.0,A
143164,1.788680e+09,9333345.0,6251.0,193.0.0.164,6251|193.0.0.164,28.263254,28.014143,28.429591,3.0,3.0,0.0,0.219745,0.0,A
143165,1.788680e+09,9333345.0,6251.0,193.0.0.164,6251|193.0.0.164,28.330417,28.283482,28.422892,3.0,3.0,0.0,0.080089,0.0,A
143166,1.788681e+09,9333345.0,6251.0,193.0.0.164,6251|193.0.0.164,28.080841,27.822658,28.373851,3.0,3.0,0.0,0.277242,0.0,A
143167,1.788681e+09,9333345.0,6251.0,193.0.0.164,6251|193.0.0.164,28.416394,28.381221,28.466356,3.0,3.0,0.0,0.044452,0.0,A


## 1. Inspeção de Integridade

In [2]:
# Verificar se tem RTT=0 
assert df_raw[df_raw['avg'] == 0].shape[0] == 0, "ERRO: RTT igual a 0 detectado."

# Verificar sobreposição A e B (não devem existir timestamps iguais para o mesmo fluxo nos dois)
df_a = df_raw[df_raw['periodo'] == 'A'].copy()
df_b = df_raw[df_raw['periodo'] == 'B'].copy()

sobreposicao = df_a.merge(df_b, on=['fluxo_id', 'timestamp'])
assert sobreposicao.shape[0] == 0, "ERRO: Sobreposição detectada entre A e B!"

print("Integridade verificada com sucesso.")

Integridade verificada com sucesso.


## 2. Cálculo do Baseline (Somente Período A)

In [3]:
baselines = []
fluxos_invalidos = []

for fluxo_id, df_fluxo in df_a.groupby('fluxo_id'):
    rtts_validos = df_fluxo['avg'].dropna()
    rtts_validos = rtts_validos[rtts_validos > 0]
    
    # Regra do RFC: 1500 válidos mínimos
    if len(rtts_validos) < 1500:
        fluxos_invalidos.append(fluxo_id)
        continue
        
    mediana = rtts_validos.median()
    mad = (rtts_validos - mediana).abs().median()
    
    jitters = df_fluxo['jitter_ms'].dropna()
    jitter_tipico = jitters.median() if len(jitters) > 0 else 0.0
    
    perda_tipica = df_fluxo['perda_pct'].median()
    prop_resposta = len(rtts_validos) / len(df_fluxo)
    
    baselines.append({
        'fluxo_id': fluxo_id,
        'mediana': mediana,
        'MAD': mad,
        'jitter_tipico': jitter_tipico,
        'perda_tipica': perda_tipica,
        'prop_resposta': prop_resposta
    })

df_baseline = pd.DataFrame(baselines)
df_baseline.to_csv('../data/processed/baseline_por_fluxo.csv', index=False)
print(f"Baselines calculados para {len(baselines)} fluxos. {len(fluxos_invalidos)} fluxos descartados (<1500 RTTs).")
display(df_baseline.head())

Baselines calculados para 69 fluxos. 0 fluxos descartados (<1500 RTTs).


,fluxo_id,mediana,MAD,jitter_tipico,perda_tipica,prop_resposta
0,6251|193.0.0.164,28.245770,0.164469,0.178789,0.0,1.0
1,6251|193.65.45.86,0.321367,0.112025,0.041823,0.0,1.0
2,6251|2001:67c:2e8:3::c100:a4,26.417991,0.153084,0.160090,0.0,1.0
3,6251|2001:998:13:1b2::2,0.368875,0.136242,0.047435,0.0,1.0
4,6251|2a02:c0:203::a,15.529405,0.157309,0.147825,0.0,1.0


## 3. Métricas Relativas e Histórico (Período B)

In [4]:
# Filtrar B para apenas fluxos com baseline válido
fluxos_validos = df_baseline['fluxo_id'].tolist()
df_b_validos = df_b[df_b['fluxo_id'].isin(fluxos_validos)].copy()
df_b_validos = df_b_validos.merge(df_baseline, on='fluxo_id', how='left')

def calcular_metricas(row):
    m = {}
    # z_robusto
    if pd.notna(row['avg']) and row['avg'] > 0:
        if row['MAD'] > 0:
            m['z_robusto'] = (row['avg'] - row['mediana']) / (1.4826 * row['MAD'])
        elif row['avg'] == row['mediana']:
            m['z_robusto'] = 0.0
        else:
            m['z_robusto'] = (row['avg'] - row['mediana']) / 1.0 # fallback
        m['aumento_pct'] = ((row['avg'] - row['mediana']) / row['mediana']) * 100.0
    else:
        m['z_robusto'] = np.nan
        m['aumento_pct'] = np.nan
        
    # jitter relativo
    if pd.notna(row['jitter_ms']) and row['jitter_ms'] > 0:
        if row['jitter_tipico'] > 0:
            m['jitter_relativo'] = row['jitter_ms'] / row['jitter_tipico']
        else:
            m['jitter_relativo'] = 3.0
    elif pd.notna(row['jitter_ms']) and row['jitter_ms'] == 0 and row['jitter_tipico'] == 0:
        m['jitter_relativo'] = 1.0
    else:
        m['jitter_relativo'] = np.nan
        
    return pd.Series(m)

df_b_validos[['z_robusto', 'aumento_pct', 'jitter_relativo']] = df_b_validos.apply(calcular_metricas, axis=1)

# Calcular os n5
df_b_validos.sort_values(by=['fluxo_id', 'timestamp'], inplace=True)

df_b_validos['n5_timeout'] = df_b_validos.groupby('fluxo_id')['timeout_atual'].rolling(5, min_periods=1).sum().reset_index(level=0, drop=True)
df_b_validos['n5_aumento80'] = df_b_validos.groupby('fluxo_id')['aumento_pct'].apply(lambda x: (x > 80).rolling(5, min_periods=1).sum()).reset_index(level=0, drop=True)

# n5_risco: cond pontual do risco
cond_risco_pontual = (
    (df_b_validos['z_robusto'] >= 2) & (df_b_validos['z_robusto'] < 3.5) |
    (df_b_validos['aumento_pct'] >= 30) & (df_b_validos['aumento_pct'] <= 80) |
    (df_b_validos['jitter_relativo'] >= 3)
)
df_b_validos['cond_risco_pontual'] = cond_risco_pontual.astype(int)
df_b_validos['n5_risco'] = df_b_validos.groupby('fluxo_id')['cond_risco_pontual'].rolling(5, min_periods=1).sum().reset_index(level=0, drop=True)

print("Métricas relativas calculadas.")

Métricas relativas calculadas.


## 4. Rotulagem (OK, RISCO, FALHA)

In [5]:
def rotular(row):
    if row['perda_pct'] >= 10:
        return 'FALHA'
    if row['n5_timeout'] >= 3:
        return 'FALHA'
    if pd.notna(row['z_robusto']) and row['z_robusto'] >= 3.5:
        return 'FALHA'
    if row['n5_aumento80'] >= 2:
        return 'FALHA'
        
    if row['cond_risco_pontual'] == 1 and row['n5_risco'] >= 2:
        return 'RISCO'
        
    return 'OK'

df_b_validos['classe'] = df_b_validos.apply(rotular, axis=1)

print("Contagem de Classes:")
print(df_b_validos['classe'].value_counts())

df_b_validos.to_csv('../data/processed/dataset_rotulado_B.csv', index=False)
print("Dataset final rotulado salvo em data/processed/dataset_rotulado_B.csv")

Contagem de Classes:
classe
OK       148948
RISCO     14042
FALHA     10684
Name: count, dtype: int64
Dataset final rotulado salvo em data/processed/dataset_rotulado_B.csv


## 5. Recorte Temporal (Treino / Validação / Teste)

In [6]:
def split_temporal(df_fluxo):
    n = len(df_fluxo)
    treino_idx = int(n * 0.50)
    val_idx = treino_idx + int(n * 0.20)
    
    # Como já está ordenado por timestamp
    df_fluxo.iloc[:treino_idx, df_fluxo.columns.get_loc('split')] = 'treino'
    df_fluxo.iloc[treino_idx:val_idx, df_fluxo.columns.get_loc('split')] = 'validacao'
    df_fluxo.iloc[val_idx:, df_fluxo.columns.get_loc('split')] = 'teste'
    return df_fluxo

df_b_validos['split'] = ''
df_b_validos = df_b_validos.groupby('fluxo_id', group_keys=False).apply(split_temporal)

print("Distribuição do Split Temporal:")
print(df_b_validos['split'].value_counts())

df_b_validos.to_csv('../data/processed/dataset_rotulado_B_split.csv', index=False)


Distribuição do Split Temporal:
split
treino       86822
teste        52149
validacao    34703
Name: count, dtype: int64
